Amazon Locker is a self-service package pickup system. A delivery driver deposits a package into an available compartment, the system generates an access token, and the customer uses that code to retrieve their package.


### QUESTIONS:

Core functions:

- How many compartments are there? 
- Sizes of compartments?
- Does driver need authentication/authorization to open the compartments? How is that done?
- What is the work flow for the driver to deliver the page? Does she just scan a code on the package before depositing it into a compartment?
- Is there expiration if package is not picked up?
- How is the token delivered to the customer?

Error conditions:
- What happens when all compartments are used? Or no right size of compartments?
- Driver opens a compartment but doesn't close it?
- What happens when customer enters wrong token, expired token or claimed token?
- What if package is not picked up?
- What about malicious attack, like force opening a compartment or many wrong token attempts?

Future extension or scope:
- UI?
- tracking? auditing?
- adding more compartments?
- customers use the locker to return packages?

### REQUIREMENTS

- Locker containing compartments of different sizes
- Driver scans code on package (my assumption), system will open an available compartment of matching size. If there is none available, error will be returned. Driver will take package back.
- Once a package is deposit, an access token will be created. Access token is unique per package.
- Customer uses token to open compartment and retrieve package. If customer enters a wrong token, expired token or used token, error is returned.
- Token will be expired after 7 days if package is not picked up. Driver will pick up expired package.

Out of Scope:
- UI
- Logic to encode/decode the token
- Tracking, auditing
- Token delivery to customers
- Mechanism to handle malicious attacks (force open or token guessing)
- Using the locker to return packages

### IMPROVEMENTS

- use number to identify each requirements
- has a main subject for each requirement and action it performs or operation on it.

### ENTITIES AND RELATIONSHIPS

Locker - the entry point and the orchestrator. It holds a collection of compartments and token mapping. Owns the package deposit and pickup.
    
Compartment - has size and state (occupied or available)

Package - has size

Token - has an access code, creation date and reference to compartment it maps to. Owns expiry validation.

### IMPROVEMENTS
- Should describe my thinking process, how I come up with the entities
- There is no need for a pakage class. All we need is an enum for size.

### CLASS DESIGN

Class Locker:
    - compartments: Compartment[]  //all compartments regardless of state
    - tokenMapping: {token.AccessCode, compartment}  //mapping for occupied compartments
    DepositPackage(Package) -> Token
    PickupPackage(Token)

Class Compartment:
    - CompartmentState: Enum(available, occupied)
    - Size: Enum(Small, Medium, Large)
    GetSize() -> CompartmentSize
    IsAvailable() -> Boolean
    Open(Token) -> Boolean
    
Class Token:
    - AccessCode
    - CreationDate
    - Compartment
    CreateNewToken(Compartment) -> Token
    IsValid() -> Boolean

### IMPROVEMENTS

- Need to include constructor for each class and the parameters for the constructor.
- Include data type for each properties

Locker Class
- Constructor missing
- tokenMapping should map the access code (string) to the token object
- The pickup method should pass in the access code (string) instead of the token object. Consider what the input is from the customer.
- I miss function for opening lockers with expired packages 

Compartment Class
- Constructor missing
- a boolean for occupacy is enough
- Open function doesn't need token to pass in
- need markOccupied() and markFree()

AccessToken Class
- missing constructor
- should have the expiration timestamp instead of creation which is determined by the Locker (orchestrator)
- Create new access code is not something the Token Class' job, it is provided from the caller. The Token class only needs to contain the access code.
- missing getCompartment(), since the Token class is responsible to keep track of all the state and information related to the access token. The Locker class doesn't own this relationship.
- missing getCode() method. Why is it needed??

In [ ]:
#IMPLEMENTATION
from enum import Enum
from datetime import datetime, timedelta
import uuid

class PackageSize(Enum):
    SMALL = 1
    MEDIUM = 2
    LARGE = 3

class CompartmentState(Enum):
    AVAILABLE = 0
    OCCUPIED = 1
    BROKEN = 2

class Compartment:
    def __init__(self, size: PackageSize):
        self._size = size
        self._state = CompartmentState.AVAILABLE
        self._id = str(uuid.uuid4())
    
    def getId(self) -> str:
        return self._id
    
    def getSize(self) -> PackageSize:
        return self._size
    
    def isAvailable(self) -> bool:
        return self._state == CompartmentState.AVAILABLE

    def markFree(self):
        self._state = CompartmentState.AVAILABLE

    def markOccupied(self):
        self._state = CompartmentState.OCCUPIED
    
    def open(self):
        pass

class Token:
    def __init__(self, accessCode: str, expiration: datetime, compartment: Compartment):
        self._accessCode = accessCode
        self._expiration = expiration
        self._compartment = compartment

    def getAccessCode(self):
        return self._accessCode
    
    def getCompartment(self):
        return self._compartment
    
    def isExpired(self):
        return datetime.now() > self._expiration

class Locker:
    def __init__(self, compartments: list[Compartment]):
        self._compartments = compartments
        self._accessTokenMapping: dict[str, Token] = {}
        
    def _generateToken(self, compartment: Compartment) -> Token:
        accessCode = str(uuid.uuid4())
        expiration = datetime.now() + timedelta(days = 7)
        return Token(accessCode, expiration, compartment)

    def _getAvailableCompartment(self, packageSize) -> Compartment:
        for compartment in self._compartments:
            if compartment.isAvailable() and compartment.getSize() == packageSize:
                return compartment

    def openExpiredCompartments(self):
        for token in self._accessTokenMapping.values():
            if token.isExpired():
                compartment = token.getCompartment()
                #should clear deposit here?
                

    def depositPackage(self, packageSize: PackageSize) -> str:
        # look for available compartment, return None if nothing is available 
        free_compartment = self._getAvailableCompartment(packageSize)
        if not free_compartment:
            return None
        
        # generate access code, map code to token
        token = self._generateToken(free_compartment)
        self._accessTokenMapping[token.getAccessCode()] = token

        # open door, update compartment state and return access code
        free_compartment.open()
        free_compartment.markOccupied()
        return token.getAccessCode()
        
    def pickup(self, accessCode: str) -> bool:
        # check validity of access token
        if (not accessCode or 
            not accessCode in self._accessTokenMapping or
            self._accessTokenMapping[accessCode].isExpired()):
            return False
        
        # open compartment
        compartment: Compartment = self._accessTokenMapping[accessCode].getCompartment()
        compartment.open()

        # update compartment state 
        compartment.markFree()

        # update accessTokenMapping
        del self._accessTokenMapping[accessCode]

        return True

# IMPROVEMENTS
# for DepositPackage()
- getAvailableCompartment() can be its own function
- token generation should be in Token class

# for pickup()
- raise seperate errors for edge conditions
- consider a clearDeposit() to encapsulate all the cleaup logic

In [24]:
compartments = [Compartment(PackageSize.SMALL), Compartment(PackageSize.LARGE)]
locker = Locker(compartments)
assert locker.depositPackage(PackageSize.MEDIUM) == None

accessCode = locker.depositPackage(PackageSize.LARGE)
assert locker.depositPackage(PackageSize.LARGE) == None

print(accessCode)
print(f"Pickup the first time: {locker.pickup(accessCode)}")
print(f"Attempt to pickup again: {locker.pickup(accessCode)}")

accessCode = locker.depositPackage(PackageSize.LARGE)
print(accessCode)
print(f"Pickup the first time: {locker.pickup(accessCode)}")
print(f"Attempt to pickup again: {locker.pickup(accessCode)}")

5a1a9e02-6697-476c-9df4-8bad0c456971
Pickup the first time: True
Attempt to pickup again: False
0de22cdc-5b65-43cf-af2a-76bbf0c0875e
Pickup the first time: True
Attempt to pickup again: False


### EXTENSIBILITY
1. "How would you handle size fallback?"
Right now if all medium compartments are full, we reject the deposit even if large compartments are available. What if we want to allow a smaller package to use a larger compartment as a fallback?

ANSWER: The ENUM that represent package size has numeric value, we can compare that value when checking for available compartments and return a compartment with larger size.

IMPROVEMENT: I should make sure to check only return the same size or the next bigger size, not any bigger size. 


2. "How would you handle compartments that are broken or under maintenance?"

ANSWER: My current design already takes care of that. I have CompartmentState enum for 'broken' or 'needs repair'. The Compartment class has function to check for availability.

3. "How would you ensure packages are actually deposited before generating access tokens?"
Right now we unlock the compartment and immediately mark it occupied, assuming the driver will deposit the package. But what if they open it and walk away? We've generated an access token for an empty compartment.

ANSWER: The compartment can install a scanner to make sure a package is deposit and expose a function to check if package has already been deposit in the compartment.

IMPROVEMENT: Should mention the key term '2-phase commit' pattern by splitting the depositPackage() into 2 operations. First driver reserves a compartment and then confirm (commit) the deposit. This involves adding additional compartment state and logic.